# Surprisal scoring: BabyLlama and Qwen-2.5-7B

Word-level surprisal in bits at the reflexive, using `minicons` (`base_two=True`).

Run from `notebooks/` after cloning the repo, or upload `data/exp1_locality.csv`, `data/exp2_hierarchy.csv`, and `data/exp3_logophoric.csv` into the Colab working directory.

In [ ]:
%pip install -q transformers accelerate minicons pandas torch

In [ ]:
from pathlib import Path

import pandas as pd
import torch
from minicons import scorer

ROOT = Path("..") if (Path("..") / "data").exists() else Path(".")
DATA = ROOT / "data"
OUT = ROOT / "results"
OUT.mkdir(parents=True, exist_ok=True)

STIMULI = [
    DATA / "exp1_locality.csv",
    DATA / "exp2_hierarchy.csv",
    DATA / "exp3_logophoric.csv",
]

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


def reflexive_surprisal(lm_model, sentence, target):
    token_scores = lm_model.token_score([sentence], surprisal=True, base_two=True)[0]
    for token, score in reversed(token_scores):
        if token.strip().replace(".", "").lower() == target.lower():
            return score
    if token_scores[-1][0].strip() == ".":
        return token_scores[-2][1]
    return token_scores[-1][1]


def score_model(model_id, outfile_tag):
    print(f"Loading {model_id}")
    lm_model = scorer.IncrementalLMScorer(model_id, device=device)
    names = [
        f"results_exp1_{outfile_tag}.csv",
        f"results_exp2_{outfile_tag}.csv",
        f"results_exp3_{outfile_tag}.csv",
    ]
    for stim_path, out_name in zip(STIMULI, names):
        df = pd.read_csv(stim_path)
        df["surprisal"] = [
            reflexive_surprisal(lm_model, row.sentence, row.target)
            for row in df.itertuples(index=False)
        ]
        out_path = OUT / out_name
        df.to_csv(out_path, index=False)
        print("wrote", out_path)

In [ ]:
score_model("babylm/babyllama-100m-2024", "babyllama")

In [ ]:
score_model("Qwen/Qwen2.5-7B", "qwen")